# RAG Retrieval Architecture & Failure Mode Analysis

**4-Pipeline Comparison: BM25 → Dense → Hybrid RRF → Hybrid + Reranking**

This notebook implements and compares four retrieval architectures on real-world documents to systematically study failure modes.

## Setup & Installation

In [ ]:
# Install dependencies
!pip install -q sentence-transformers rank-bm25 faiss-cpu datasets PyMuPDF
!pip install -q transformers torch accelerate bitsandbytes
!pip install -q matplotlib seaborn pandas numpy tqdm

In [ ]:
# Clone repository (if using GitHub)
# !git clone https://github.com/YOUR_USERNAME/RetrieveAudit.git
# %cd RetrieveAudit

In [ ]:
# Alternatively, upload files directly
import sys
from pathlib import Path

# Create directory structure
!mkdir -p src/retrieval src/evaluation src/data_processing data results

## 1. Load Dataset

Using **FinanceBench** - real SEC 10-K filings with challenging numerical reasoning questions.

In [ ]:
from datasets import load_dataset

# Load FinanceBench dataset
print("Loading FinanceBench dataset...")
dataset = load_dataset("PatronusAI/financebench")

# Inspect structure
print(f"\nDataset splits: {list(dataset.keys())}")
print(f"Number of examples: {len(dataset['train'])}")
print(f"\nExample question:")
print(dataset['train'][0])

In [ ]:
# Extract Q&A pairs and documents
import json

def prepare_financebench_data(dataset, n_samples=100):
    """
    Convert FinanceBench to our format.
    
    Returns:
        documents: List of dicts with doc_id and text
        qa_pairs: List of dicts with query_id, query, answer, relevant_chunks
    """
    documents = []
    qa_pairs = []
    
    seen_docs = set()
    
    for i, example in enumerate(dataset['train'][:n_samples]):
        # Extract document
        doc_id = example.get('doc_name', f'doc_{i}')
        doc_text = example.get('evidence', '')
        
        if doc_id not in seen_docs and doc_text:
            documents.append({
                'doc_id': doc_id,
                'text': doc_text
            })
            seen_docs.add(doc_id)
        
        # Extract Q&A
        qa_pairs.append({
            'query_id': f'q{i:04d}',
            'query': example.get('question', ''),
            'answer': example.get('answer', ''),
            'doc_id': doc_id,
            'relevant_chunks': []  # Will be populated after chunking
        })
    
    return documents, qa_pairs

documents, qa_pairs = prepare_financebench_data(dataset, n_samples=150)
print(f"Prepared {len(documents)} documents and {len(qa_pairs)} Q&A pairs")

## 2. Document Chunking

**Fixed parameters for fair comparison**: 512 tokens, 64-token overlap

In [ ]:
# Paste chunker.py code here or import if uploaded
# For brevity, assuming code is available

from src.data_processing.chunker import DocumentChunker

# Initialize chunker
chunker = DocumentChunker(chunk_size=512, overlap=64)

# Chunk all documents
all_chunks = chunker.chunk_documents(documents)

print(f"Created {len(all_chunks)} chunks from {len(documents)} documents")
print(f"\nExample chunk:")
print(f"  ID: {all_chunks[0].chunk_id}")
print(f"  Doc: {all_chunks[0].doc_id}")
print(f"  Tokens: {all_chunks[0].metadata['token_count']}")
print(f"  Text: {all_chunks[0].text[:150]}...")

In [ ]:
# Convert to dict format for retrievers
chunks_dict = [
    {
        'chunk_id': c.chunk_id,
        'doc_id': c.doc_id,
        'text': c.text
    }
    for c in all_chunks
]

## 3. Initialize All 4 Retrievers

In [ ]:
from src.retrieval.sparse_retriever import SparseRetriever
from src.retrieval.dense_retriever import DenseRetriever
from src.retrieval.hybrid_retriever import HybridRetriever
from src.retrieval.reranker import HybridWithReranker

print("Initializing retrievers...\n")

# 1. Sparse (BM25)
print("[1/4] BM25...")
sparse = SparseRetriever(chunks_dict)

# 2. Dense
print("\n[2/4] Dense Retriever...")
dense = DenseRetriever(
    chunks_dict, 
    model_name="BAAI/bge-base-en-v1.5",
    device="cuda"  # Use GPU in Colab
)

# 3. Hybrid RRF
print("\n[3/4] Hybrid (RRF)...")
hybrid = HybridRetriever(sparse, dense, k=60)

# 4. Hybrid + Reranking
print("\n[4/4] Hybrid + Reranker...")
hybrid_reranked = HybridWithReranker(
    hybrid,
    reranker_model="BAAI/bge-reranker-base"
)

print("\n✅ All retrievers ready!")

## 4. Run Comparison

In [ ]:
# Test on a single query first
test_query = qa_pairs[0]['query']
print(f"Test Query: {test_query}\n")

print("BM25 Results:")
bm25_res = sparse.retrieve(test_query, top_k=3)
for r in bm25_res:
    print(f"  Rank {r['rank']}: {r['chunk_id']} (score: {r['score']:.3f})")

print("\nDense Results:")
dense_res = dense.retrieve(test_query, top_k=3)
for r in dense_res:
    print(f"  Rank {r['rank']}: {r['chunk_id']} (score: {r['score']:.3f})")

print("\nHybrid RRF Results:")
hybrid_res = hybrid.retrieve(test_query, top_k=3)
for r in hybrid_res:
    print(f"  Rank {r['rank']}: {r['chunk_id']} (RRF: {r['rrf_score']:.4f})")

print("\nHybrid + Reranked Results:")
reranked_res = hybrid_reranked.retrieve(test_query, top_k=3, retrieval_depth=10)
for r in reranked_res:
    print(f"  Rank {r['rank']}: {r['chunk_id']} (rerank: {r['rerank_score']:.4f})")

In [ ]:
# Run full comparison (this will take a while)
from tqdm.auto import tqdm

def run_full_comparison(qa_pairs, top_k=5):
    results = {
        'bm25': [],
        'dense': [],
        'hybrid_rrf': [],
        'hybrid_reranked': []
    }
    
    queries = [qa['query'] for qa in qa_pairs]
    query_ids = [qa['query_id'] for qa in qa_pairs]
    
    # BM25
    print("Running BM25...")
    bm25_results = sparse.batch_retrieve(queries, top_k=top_k)
    results['bm25'] = [
        {'query_id': qid, 'retrieved_ids': [r['chunk_id'] for r in res]}
        for qid, res in zip(query_ids, bm25_results)
    ]
    
    # Dense
    print("Running Dense...")
    dense_results = dense.batch_retrieve(queries, top_k=top_k)
    results['dense'] = [
        {'query_id': qid, 'retrieved_ids': [r['chunk_id'] for r in res]}
        for qid, res in zip(query_ids, dense_results)
    ]
    
    # Hybrid
    print("Running Hybrid RRF...")
    hybrid_results = hybrid.batch_retrieve(queries, top_k=top_k, retrieval_depth=20)
    results['hybrid_rrf'] = [
        {'query_id': qid, 'retrieved_ids': [r['chunk_id'] for r in res]}
        for qid, res in zip(query_ids, hybrid_results)
    ]
    
    # Reranked
    print("Running Hybrid + Reranking...")
    reranked_results = hybrid_reranked.batch_retrieve(queries, top_k=top_k, retrieval_depth=20)
    results['hybrid_reranked'] = [
        {'query_id': qid, 'retrieved_ids': [r['chunk_id'] for r in res]}
        for qid, res in zip(query_ids, reranked_results)
    ]
    
    return results

# Run on all queries
all_results = run_full_comparison(qa_pairs[:100], top_k=5)
print("\n✅ Comparison complete!")

## 5. Evaluation & Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Create comparison DataFrame
# NOTE: You need ground truth relevant_chunks for real evaluation
# This is a placeholder for demonstration

comparison_data = {
    'Retriever': ['BM25', 'Dense', 'Hybrid RRF', 'Hybrid+Rerank'],
    'Recall@5': [0.52, 0.58, 0.66, 0.74],  # Placeholder values
    'MRR': [0.45, 0.51, 0.60, 0.68],
    'Precision@5': [0.38, 0.42, 0.48, 0.56]
}

df = pd.DataFrame(comparison_data)

# Plot comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, metric in enumerate(['Recall@5', 'MRR', 'Precision@5']):
    ax = axes[idx]
    bars = ax.bar(df['Retriever'], df[metric], 
                   color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4'])
    ax.set_ylabel(metric, fontsize=12)
    ax.set_ylim(0, 1.0)
    ax.set_title(metric, fontsize=14, fontweight='bold')
    
    # Add value labels
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}', ha='center', va='bottom')
    
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('retrieval_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n📊 Comparison Chart:")
print(df.to_string(index=False))

## 6. Error Analysis Framework

In [ ]:
from src.evaluation.error_taxonomy import ErrorAnalyzer, FailureMode, ErrorAnnotation

# Initialize analyzer
analyzer = ErrorAnalyzer()

# Example: Manually annotate a failed case
# In practice, you'll loop through failed queries

example_error = ErrorAnnotation(
    query_id="q0001",
    query="What was Company X's total revenue in Q4 2024?",
    generated_answer="Company X had strong performance in 2024.",
    ground_truth_answer="Total revenue was $5.2 billion in Q4 2024.",
    retrieved_chunks=[
        {'chunk_id': 'doc1_chunk_8', 'text': 'Operations expanded...', 'rank': 1},
        {'chunk_id': 'doc1_chunk_15', 'text': 'Total revenue was $5.2B in Q4 2024.', 'rank': 2}
    ],
    failure_mode=FailureMode.LOST_IN_THE_MIDDLE,
    failure_description="Correct chunk at rank 2, but LLM generated vague answer.",
    correct_chunks_retrieved=True,
    correct_chunk_positions=[1],
    retriever_type="bm25"
)

analyzer.add_annotation(example_error)

# Generate report
print(analyzer.generate_report())

## Next Steps

1. **Ground Truth Labeling**: Manually label 50-100 failed cases with failure modes
2. **Ablation Study**: Test chunk sizes (256, 512, 1024)
3. **Visualization**: Create stacked bar charts showing failure mode reduction
4. **Documentation**: Write up findings for GitHub README

**Expected Timeline**: Days 5-7 of the 8-day plan